First, future climate data is downloaded, one emission scenario at a time due to limited computer storage.

In [1]:
import requests
import os
import time
from bs4 import BeautifulSoup

In [ ]:
username = '' # removed
password = '' # removed

base_output = r'C:\Users\alexd\OneDrive\Documents\Uni\EMDA\Dissertation\data + code\climate_data'

variables = ['sfcWind']
scenarios = ['rcp85']  # only rcp85 this time

session = requests.Session()
session.auth = (username, password)

def download_with_retry(session, url, filepath, max_retries=5, chunk_size=8192):
    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(url, stream=True, timeout=60)
            r.raise_for_status()
            with open(filepath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if chunk:
                        f.write(chunk)

            # verify size against what the server reported, if available
            expected = r.headers.get('Content-Length')
            if expected is not None and os.path.getsize(filepath) != int(expected):
                raise IOError(f"Size mismatch: got {os.path.getsize(filepath)}, expected {expected}")

            return True

        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                IOError) as e:
            print(f"  Attempt {attempt}/{max_retries} failed: {e}")
            if os.path.exists(filepath):
                os.remove(filepath)  # remove partial file before retrying
            time.sleep(2 ** attempt)  # backoff: 2s, 4s, 8s, 16s, 32s

    print(f"  FAILED after {max_retries} attempts: {filepath}")
    return False

for scenario in scenarios:
    for var in variables:
        base_url = f'https://dap.ceda.ac.uk/badc/deposited2021/chess-scape/data/{scenario}_bias-corrected/01/daily/{var}/'

        output_dir = os.path.join(base_output, f'{var}_future_{scenario}')
        os.makedirs(output_dir, exist_ok=True)

        response = session.get(base_url)
        soup = BeautifulSoup(response.text, 'html.parser')

        files = []
        for link in soup.find_all('a'):
            href = link.get('href', '')
            if href.endswith('.nc'):
                try:
                    year = int(href.split('daily_')[1][:4])
                    if 2027 <= year <= 2080:
                        files.append(href)
                except:
                    pass

        print(f"\n{scenario} {var}: {len(files)} files to download")

        for filename in files:
            url = base_url + filename
            filepath = os.path.join(output_dir, filename)

            if os.path.exists(filepath):
                print(f"Skipping: {filename}")
                continue

            print(f"Downloading: {scenario}/{var}/{filename}")
            download_with_retry(session, url, filepath)

print("\nAll downloads complete")


rcp85 sfcWind: 647 files to download
Downloading: rcp85/sfcWind/chess-scape_rcp85_bias-corrected_01_sfcWind_uk_1km_daily_20270101-20270130.nc
Downloading: rcp85/sfcWind/chess-scape_rcp85_bias-corrected_01_sfcWind_uk_1km_daily_20270201-20270230.nc
Downloading: rcp85/sfcWind/chess-scape_rcp85_bias-corrected_01_sfcWind_uk_1km_daily_20270301-20270330.nc
Downloading: rcp85/sfcWind/chess-scape_rcp85_bias-corrected_01_sfcWind_uk_1km_daily_20270401-20270430.nc
Downloading: rcp85/sfcWind/chess-scape_rcp85_bias-corrected_01_sfcWind_uk_1km_daily_20270501-20270530.nc
Downloading: rcp85/sfcWind/chess-scape_rcp85_bias-corrected_01_sfcWind_uk_1km_daily_20270601-20270630.nc
Downloading: rcp85/sfcWind/chess-scape_rcp85_bias-corrected_01_sfcWind_uk_1km_daily_20270701-20270730.nc
Downloading: rcp85/sfcWind/chess-scape_rcp85_bias-corrected_01_sfcWind_uk_1km_daily_20270801-20270830.nc
Downloading: rcp85/sfcWind/chess-scape_rcp85_bias-corrected_01_sfcWind_uk_1km_daily_20270901-20270930.nc
Downloading: rcp8

Next the future projection data is combined with hospital sites, so that each site has daily values for each rcp.

In [28]:
import pandas as pd

df_energy = pd.read_parquet('sense_acute_climate.parquet')

In [29]:
# re-extract coordinates for sites

import pgeocode
from pyproj import Transformer

nomi = pgeocode.Nominatim('gb')
postcodes = df_energy.drop_duplicates('site_code')[['site_code', 'postcode']].copy()
postcodes['postcode'] = postcodes['postcode'].str.strip()

geo = nomi.query_postal_code(postcodes['postcode'].tolist())
postcodes['latitude'] = geo['latitude'].values
postcodes['longitude'] = geo['longitude'].values

transformer = Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True)

sites_bng = {}
for _, row in postcodes.iterrows():
    x, y = transformer.transform(row['longitude'], row['latitude'])
    sites_bng[row['site_code']] = {'x': x, 'y': y}

print(f"Sites defined: {len(sites_bng)}")

Sites defined: 35


In [ ]:
import xarray as xr
import glob

base_output = r'C:\Users\alexd\OneDrive\Documents\Uni\EMDA\Dissertation\data + code\climate_data'
scenario = 'rcp85'
variables = ['sfcWind']

# extract all variables and merge into one dataframe
all_var_dfs = []

for var in variables:
    folder = os.path.join(base_output, f'{var}_future_{scenario}')
    files = sorted(glob.glob(os.path.join(folder, '*.nc')))
    print(f"\n{var}: {len(files)} files found")
    
    records = []
    
    for i, fpath in enumerate(files):
        ds = xr.open_dataset(fpath, decode_times=True, use_cftime=True)
        
        for site_code, coord in sites_bng.items():
            point = ds.sel(x=coord['x'], y=coord['y'], method='nearest')
            df = point[[var]].to_dataframe().reset_index()
            df['site_code'] = site_code
            records.append(df[['time', 'site_code', var]])
        
        ds.close()
        
    
    var_df = pd.concat(records, ignore_index=True)
    var_df['year'] = var_df['time'].apply(lambda t: t.year)
    var_df['month'] = var_df['time'].apply(lambda t: t.month)
    var_df['day'] = var_df['time'].apply(lambda t: t.day)
    var_df = var_df.drop(columns=['time'])
    var_df = var_df.rename(columns={var: var})
    
    # convert k to c
    if var in ['tas', 'tasmax', 'tasmin'] and var_df[var].mean() > 100:
        var_df[var] = var_df[var] - 273.15
    
    all_var_dfs.append(var_df)
    print(f"  {var} done")

# merge all variables into one dataframe
print("\nMerging variables...")

## this is code to create parquet, now since we added new variables we must merge with already existing parquet


#climate_rcp45 = all_var_dfs[0]
#for var_df in all_var_dfs[1:]:
#    climate_rcp45 = climate_rcp45.merge(var_df, on=['site_code', 'year', 'month', 'day'], how='inner')

#print(f"Final shape: {climate_rcp45.shape}")
#print(climate_rcp45.head())
#print(climate_rcp45.describe())

# save
#climate_rcp45.to_parquet(os.path.join(base_output, 'climate_rcp45_sitelevel.parquet'), index=False)

existing = pd.read_parquet(os.path.join(base_output, 'climate_rcp85_sitelevel.parquet'))
updated = existing.merge(all_var_dfs[0], on=['site_code', 'year', 'month', 'day'], how='left')
#updated = updated.merge(all_var_dfs[1], on=['site_code', 'year', 'month', 'day'], how='left') # use this if 2 variables being added

print(f"Updated shape: {updated.shape}")
print(updated.isnull().sum())

updated.to_parquet(os.path.join(base_output, 'climate_rcp85_sitelevel.parquet'), index=False)
print("Saved successfully")


sfcWind: 647 files found


C:\Users\alexd\AppData\Local\Temp\ipykernel_15564\737542691.py:19: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(fpath, decode_times=True, use_cftime=True)
C:\Users\alexd\AppData\Local\Temp\ipykernel_15564\737542691.py:19: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(fpath, decode_times=True, use_cftime=True)
C:\Users\alexd\AppData\Local\Temp\ipykernel_15564\737542691.py:19: DeprecationWarning: Usage of 'use_cftime' as a kwarg is depreca

  sfcWind done

Merging variables...
Updated shape: (679350, 10)
site_code    0
tas          0
year         0
month        0
day          0
tasmin       0
tasmax       0
hurs         0
rsds         0
sfcWind      0
dtype: int64
Saved successfully


In [2]:
import pandas as pd
import os

base_output = r'C:\Users\alexd\OneDrive\Documents\Uni\EMDA\Dissertation\data + code\climate_data'

for scenario in ['rcp26', 'rcp45', 'rcp85']:
    print(f"{scenario.upper()}")
    
    path = os.path.join(base_output, f'climate_{scenario}_sitelevel.parquet')
    
    if not os.path.exists(path):
        print("File not found")
        continue
    
    df = pd.read_parquet(path)
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.describe().round(2))

RCP26
Shape: (679350, 10)
Columns: ['site_code', 'tas', 'year', 'month', 'day', 'tasmin', 'tasmax', 'hurs', 'rsds', 'sfcWind']
             tas       year      month        day     tasmin     tasmax  \
count  679350.00  679350.00  679350.00  679350.00  679350.00  679350.00   
mean       11.52    2053.46       6.49      15.50       8.34      14.63   
std         5.48      15.56       3.45       8.66       4.92       6.32   
min       -10.42    2027.00       1.00       1.00     -22.46      -5.67   
25%         7.37    2040.00       3.00       8.00       4.83       9.79   
50%        11.23    2053.00       6.00      15.50       8.33      14.04   
75%        15.93    2067.00       9.00      23.00      12.24      19.50   
max        31.32    2080.00      12.00      30.00      25.30      40.63   

            hurs       rsds    sfcWind  
count  679350.00  679350.00  679350.00  
mean       80.53     127.96       4.06  
std         9.67      87.64       1.83  
min        30.02       1.64      